## Empresa: FinNova Consulting SpA

Agente : FinNova AI Knowledge Assistant

In [ ]:
#==========================================================
#INSTALACION DE LIBRERIAS
#Proyecto: FinNova AI Knowledge Assistant
#==========================================================

!pip -q install langchain
!pip -q install langchain-community
!pip -q install langchain-google-genai
!pip -q install google-generativeai
!pip -q install pypdf
!pip -q install faiss-cpu
!pip -q install python-dotenv
!pip -q install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 85.0 MB/s eta 0:00:00


In [ ]:
# ==========================================================
# IMPORTACIÓN DE LIBRERÍAS
# ==========================================================

import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.vectorstores import FAISS

/tmp/ipykernel_1136/3334280882.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
# ==========================================================
# CONFIGURACIÓN SEGURA DE LA API KEY DESDE GOOGLE COLAB
# ==========================================================

from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
# ==========================================================
# INICIALIZACIÓN DEL MODELO GEMINI
# ==========================================================

from langchain_google_genai import ChatGoogleGenerativeAI

MODEL_NAME = "gemini-3.6-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,

)

In [ ]:
def obtener_texto_respuesta(respuesta):
    if isinstance(respuesta.content, str):
        return respuesta.content

    if isinstance(respuesta.content, list):
        return "".join(
            bloque.get("text", "")
            for bloque in respuesta.content
            if isinstance(bloque, dict)
        )

    return str(respuesta.content)



In [ ]:
# ==========================================================
# VERIFICACIÓN DEL ENTORNO
# ==========================================================

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GOOGLE_API_KEY"))

print("Modelos disponibles:\n")

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print("-", model.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Modelos disponibles:

- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemma-4-26b-a4b-it
- models/gemma-4-31b-it
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3.1-flash-lite
- models/gemini-3-pro-image-preview
- models/gemini-3-pro-image
- models/nano-banana-pro-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-3.1-flash-image
- models/gemini-3.1-flash-lite-image
- models/gemini-3.5-flash
- models/gemini-3.5-flash-lite
- models/gemini-omni-flash-preview
- models/gemini-3.6-f

In [ ]:
# ==========================================================
# LECTURA DE LOS DOCUMENTOS PDF
# ==========================================================

import os
from langchain_community.document_loaders import PyPDFLoader

documentos = []

carpeta = "data"

for archivo in os.listdir(carpeta):

    if archivo.endswith(".pdf"):

        ruta = os.path.join(carpeta, archivo)

        print(f"Cargando: {archivo}")

        loader = PyPDFLoader(ruta)

        documentos.extend(loader.load())

print("--------------------------------")
print(f"Total de páginas: {len(documentos)}")

Cargando: Términos y Condiciones de Uso.pdf
Cargando: Manual SAP Business One para Finanzas.pdf
Cargando: Manual_Financiero_Corporativo.pdf
--------------------------------
Total de páginas: 22


In [ ]:
# ==========================================================
# DIVISIÓN DE DOCUMENTOS EN CHUNKS
# ==========================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documentos)

print(f"Cantidad de chunks generados: {len(chunks)}")

Cantidad de chunks generados: 36


In [ ]:
# ==========================================================
# CREACIÓN DEL MODELO DE EMBEDDINGS
# ==========================================================

from langchain_google_genai import GoogleGenerativeAIEmbeddings

EMBEDDING_MODEL = "models/gemini-embedding-2"

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBEDDING_MODEL
)

print("Modelo de embeddings cargado correctamente.")

Modelo de embeddings cargado correctamente.


In [ ]:
# ==========================================================
# CREACIÓN DE LA BASE VECTORIAL FAISS
# ==========================================================

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("Base vectorial creada correctamente.")

Base vectorial creada correctamente.


In [ ]:
# ==========================================================
# GUARDAR EL ÍNDICE FAISS
# ==========================================================

vectorstore.save_local("finnova_vector_db")

print("Base vectorial almacenada correctamente.")

Base vectorial almacenada correctamente.


In [ ]:
# ==========================================================
# CARGAR BASE VECTORIAL EXISTENTE
# ==========================================================

vectorstore = FAISS.load_local(
    "finnova_vector_db",
    embeddings,
    allow_dangerous_deserialization=True
)

print("Base vectorial cargada.")

Base vectorial cargada.


In [ ]:
# ==========================================================
# CREACIÓN DEL RETRIEVER
# ==========================================================

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever creado correctamente.")

Retriever creado correctamente.


In [ ]:
# ==========================================================
# PRUEBA DEL RETRIEVER
# ==========================================================

consulta = "¿Cuál es el flujo de aprobación de pagos?"

resultados = retriever.invoke(consulta)

print(f"Se encontraron {len(resultados)} fragmentos.\n")

for i, doc in enumerate(resultados, start=1):
    print(f"========== Fragmento {i} ==========")
    print(doc.page_content[:500])
    print("\n")

Se encontraron 4 fragmentos.

========== Fragmento 1 ==========
Toda  reposición  deberá  contar  con  comprobantes  válidos.  
 
11.  Flujo  de  Aprobación  de  Pagos  
Los  pagos  deberán  seguir  el  siguiente  flujo:  
Proveedor


========== Fragmento 2 ==========
↓  
Recepción  de  factura  
↓  
Validación  tributaria  
↓  
Registro  en  SAP  Business  One  
↓  
Aprobación  del  responsable  del  área  
↓  
Aprobación  del  Gerente  de  Finanzas  
↓  
Programación  bancaria  
↓  
Pago  
↓  
Archivo  documental  
 
12.  Cierre  Contable  Mensual  
Al  cierre  de  cada  mes  deberán  ejecutarse  las  siguientes  actividades:  
●  Registro  de  todas  las  facturas.  ●  Registro  de  ingresos.  ●  Conciliaciones  bancarias.  ●  Depreciaciones.  ●  Provision


========== Fragmento 3 ==========
6.  Flujo  de  Caja  
El  flujo  de  caja  proyectado  deberá  actualizarse  semanalmente.  
La  planificación  considera:  
●  Cobros  esperados.  ●  Pagos  comprometidos.  ●  Obligaciones  tri

Creacion del Prompt - Agente FinNova

In [ ]:
# ==========================================================
# PROMPT DEL ASISTENTE
# ==========================================================

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Eres FinNova AI Knowledge Assistant.

Tu función es responder únicamente utilizando la información disponible
en los documentos corporativos proporcionados.

Reglas:

- Responde únicamente con información encontrada en el contexto.
- Si la información no existe en los documentos responde:

"No encontré esa información en la documentación disponible."

- No inventes respuestas.
- Responde en español.
- Sé claro y profesional.

Contexto:

{context}

Pregunta:

{input}
""")

In [ ]:
import langchain

print(langchain.__version__)

1.3.13


In [ ]:
# ==========================================================
# PROMPT DEL AGENTE RAG
# ==========================================================

SYSTEM_PROMPT = """
Eres FinNova AI Knowledge Assistant.

Tu función es responder únicamente utilizando la información presente
en los documentos corporativos de FinNova Consulting SpA.

Reglas importantes:

1. Responde solamente con información del contexto.
2. No inventes información.
3. Si la respuesta no está en los documentos responde:

"No encontré esa información en la documentación disponible."

4. Responde en español.
5. Sé claro, profesional y preciso.

Contexto:

{context}

Pregunta:

{question}

Respuesta:
"""

In [ ]:
# ==========================================================
# RECUPERAR CONTEXTO Y FUENTES
# ==========================================================

def obtener_contexto(pregunta, k=4):

    documentos = retriever.invoke(pregunta)

    contexto = "\n\n".join(
        doc.page_content for doc in documentos
    )

    fuentes = list(
        set(
            doc.metadata.get("source", "Documento desconocido")
            for doc in documentos
        )
    )

    return contexto, fuentes

In [ ]:
import time

# ==========================================================
# CONSULTA AL AGENTE
# ==========================================================

def consultar_agente(pregunta):

    inicio = time.time()

    contexto, fuentes = obtener_contexto(pregunta)

    prompt = SYSTEM_PROMPT.format(
        context=contexto,
        question=pregunta
    )

    respuesta = llm.invoke(prompt)

    tiempo = time.time() - inicio

    return {
        "pregunta": pregunta,
        "respuesta": obtener_texto_respuesta(respuesta),
        "fuentes": fuentes,
        "tiempo": tiempo
    }

In [ ]:
# ==========================================================
# MOSTRAR RESPUESTA
# ==========================================================

def mostrar_respuesta(resultado):

    print("=" * 70)
    print("🤖 FinNova AI Knowledge Assistant")
    print("=" * 70)

    print(f"\n📌 Pregunta:\n{resultado['pregunta']}")

    print("\n💬 Respuesta:\n")
    print(resultado["respuesta"])

    print("\n📚 Documento(s) consultado(s):")

    for fuente in resultado["fuentes"]:
        nombre = fuente.split("/")[-1].replace(".pdf", "")
        print(f"   ✓ {nombre}")

    print(f"\n⏱ Tiempo de respuesta: {resultado['tiempo']:.2f} segundos")

    print("\n" + "=" * 70)

PRUEBAS CONSULTA A AGENTE

In [ ]:
respuesta = llm.invoke("Responde únicamente con la palabra OK")

print(obtener_texto_respuesta(respuesta))

OK


In [ ]:
resultado = consultar_agente(
    "¿Cuál es el flujo de aprobación de pagos?"
)

mostrar_respuesta(resultado)

🤖 FinNova AI Knowledge Assistant

📌 Pregunta:
¿Cuál es el flujo de aprobación de pagos?

💬 Respuesta:

El flujo de aprobación de pagos debe seguir las siguientes etapas:

1. Proveedor
2. Recepción de factura
3. Validación tributaria
4. Registro en SAP Business One
5. Aprobación del responsable del área
6. Aprobación del Gerente de Finanzas
7. Programación bancaria
8. Pago
9. Archivo documental

📚 Documento(s) consultado(s):
   ✓ Manual_Financiero_Corporativo

⏱ Tiempo de respuesta: 2.51 segundos



In [ ]:
# ==========================================================
# VALIDACIÓN DEL AGENTE
# ==========================================================

preguntas_prueba = [

    # Manual Financiero
    "¿Cuál es el flujo de aprobación de pagos?",
    "¿Quién aprueba pagos superiores a USD 10.000?",
    "¿Qué documentos respaldan un pago?",

    # SAP Business One
    "¿Qué es un Socio de Negocio?",
    "¿Cómo registrar una factura de proveedor?",
    "¿Qué módulo administra a los proveedores?",

    # Términos y Condiciones
    "¿Quién puede acceder a FinNova AI?",
    "¿Cómo se protegen los datos de la empresa?",
    "¿Qué ocurre ante un uso indebido del sistema?"
]

for i, pregunta in enumerate(preguntas_prueba, start=1):

    print(f"\n{'='*80}")
    print(f"PRUEBA {i}")
    print(f"{'='*80}\n")

    resultado = consultar_agente(pregunta)

    mostrar_respuesta(resultado)


PRUEBA 1

🤖 FinNova AI Knowledge Assistant

📌 Pregunta:
¿Cuál es el flujo de aprobación de pagos?

💬 Respuesta:

El flujo de aprobación de pagos debe seguir la siguiente secuencia:

1. Proveedor
2. Recepción de factura
3. Validación tributaria
4. Registro en SAP Business One
5. Aprobación del responsable del área
6. Aprobación del Gerente de Finanzas
7. Programación bancaria
8. Pago
9. Archivo documental

📚 Documento(s) consultado(s):
   ✓ Manual_Financiero_Corporativo

⏱ Tiempo de respuesta: 2.86 segundos


PRUEBA 2

🤖 FinNova AI Knowledge Assistant

📌 Pregunta:
¿Quién aprueba pagos superiores a USD 10.000?

💬 Respuesta:

No encontré esa información en la documentación disponible.

📚 Documento(s) consultado(s):
   ✓ Manual_Financiero_Corporativo

⏱ Tiempo de respuesta: 3.02 segundos


PRUEBA 3

🤖 FinNova AI Knowledge Assistant

📌 Pregunta:
¿Qué documentos respaldan un pago?

💬 Respuesta:

De acuerdo con la documentación disponible:

* Todo pago debe quedar asociado a la **factura cor